# Phase 1: LogFile Raw Data Parsing

## Overview
This notebook parses the NTFS $LogFile to extract transaction records for timestomping detection analysis.

### Purpose
Extract the following columns from $LogFile:
- **LSN**: Log Sequence Number (record identifier)
- **Redo OP**: Redo operation code
- **Undo OP**: Undo operation code
- **Record Offset**: Offset within the target MFT record
- **Attribute Offset**: Offset within the attribute being modified
- **Undo Data - Timestamps**: Before timestamps ($SI-C/M/E/A)
- **Redo Data - Timestamps**: After timestamps ($SI-C/M/E/A)
- **Target VCN**: Virtual Cluster Number (location of target MFT entry)
- **MFT Cluster Index**: Index within the cluster
- **Target FRN**: Calculated File Reference Number

### Detection Relevance
The LogFile is critical for detecting timestamp manipulation because:
1. **UpdateResidentValue (0x7)** operations on $SI show timestamp changes
2. Redo vs Undo data comparison reveals before/after timestamps
3. Can detect $SI-C and $SI-M manipulation directly
4. Cross-reference with MFT using LSN

### Input
- Raw $LogFile: `data/raw/PE/01-PE/$LogFile`

### Output
- Parsed CSV: `data/Phase 1: Raw Data Parsing/01-PE/LogFile.csv`

### Library
Uses [dfir_ntfs](https://github.com/bamonskiy-kaban/dfir_ntfs) for NTFS artifact parsing.

---
**Reference**: Oh, Lee, and Hwang (2024) - Algorithms 1-4 (LogFile-1A)


In [13]:
# [Cell 1] Install and Import Dependencies
# Install dfir_ntfs if not already installed
import subprocess
import sys

def install_package(package):
    """Install a package using pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install dfir_ntfs from GitHub
try:
    import dfir_ntfs
    print(f"dfir_ntfs version: {dfir_ntfs.__version__}")
except ImportError:
    print("Installing dfir_ntfs...")
    install_package("git+https://github.com/bamonskiy-kaban/dfir_ntfs.git")
    import dfir_ntfs
    print(f"dfir_ntfs installed successfully. Version: {dfir_ntfs.__version__}")


dfir_ntfs version: 1.1.19


In [14]:
# [Cell 2] Import Required Libraries
import os
import struct
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

# dfir_ntfs modules
from dfir_ntfs.LogFile import LogFileParser

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully.")


Libraries imported successfully.


In [15]:
# [Cell 3] Define Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input path - raw LogFile
INPUT_LOGFILE = PROJECT_DIR / "data" / "raw" / "PE" / "01-PE" / "$LogFile"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing" / "01-PE"
OUTPUT_CSV = OUTPUT_DIR / "LogFile.csv"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input file exists
if INPUT_LOGFILE.exists():
    file_size_mb = INPUT_LOGFILE.stat().st_size / (1024 * 1024)
    print(f"Input LogFile: {INPUT_LOGFILE}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"LogFile not found: {INPUT_LOGFILE}")

print(f"Output will be saved to: {OUTPUT_CSV}")


Input LogFile: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$LogFile
File size: 64.00 MB
Output will be saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/LogFile.csv


In [16]:
# [Cell 3] Define Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input path - raw LogFile
INPUT_LOGFILE = PROJECT_DIR / "data" / "raw" / "PE" / "01-PE" / "$LogFile"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing" / "01-PE"
OUTPUT_CSV = OUTPUT_DIR / "LogFile.csv"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input file exists
if INPUT_LOGFILE.exists():
    file_size_mb = INPUT_LOGFILE.stat().st_size / (1024 * 1024)
    print(f"Input LogFile: {INPUT_LOGFILE}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"LogFile not found: {INPUT_LOGFILE}")

print(f"Output will be saved to: {OUTPUT_CSV}")


Input LogFile: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$LogFile
File size: 64.00 MB
Output will be saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/LogFile.csv


In [17]:
# [Cell 4] Define Operation Codes and Helper Functions

# NTFS Operation Codes
OPERATION_CODES = {
    0x00: "Noop",
    0x01: "CompensationLogRecord",
    0x02: "InitializeFileRecordSegment",
    0x03: "DeallocateFileRecordSegment",
    0x04: "WriteEndOfFileRecordSegment",
    0x05: "CreateAttribute",
    0x06: "DeleteAttribute",
    0x07: "UpdateResidentValue",
    0x08: "UpdateNonresidentValue",
    0x09: "UpdateMappingPairs",
    0x0A: "DeleteDirtyClusters",
    0x0B: "SetNewAttributeSizes",
    0x0C: "AddIndexEntryRoot",
    0x0D: "DeleteIndexEntryRoot",
    0x0E: "AddIndexEntryAllocation",
    0x0F: "DeleteIndexEntryAllocation",
    0x10: "WriteEndOfIndexBuffer",
    0x11: "SetIndexEntryVcnRoot",
    0x12: "SetIndexEntryVcnAllocation",
    0x13: "UpdateFileNameRoot",
    0x14: "UpdateFileNameAllocation",
    0x15: "SetBitsInNonresidentBitMap",
    0x16: "ClearBitsInNonresidentBitMap",
    0x17: "HotFix",
    0x18: "EndTopLevelAction",
    0x19: "PrepareTransaction",
    0x1A: "CommitTransaction",
    0x1B: "ForgetTransaction",
    0x1C: "OpenNonresidentAttribute",
    0x1D: "OpenAttributeTableDump",
    0x1E: "AttributeNamesDump",
    0x1F: "DirtyPageTableDump",
    0x20: "TransactionTableDump",
    0x21: "UpdateRecordDataRoot",
    0x22: "UpdateRecordDataAllocation",
    0x23: "UpdateRelativeDataIndex",
    0x24: "UpdateRelativeDataAllocation",
    0x25: "ZeroEndOfFileRecord"
}

def get_operation_name(op_code):
    """Get human-readable operation name."""
    return OPERATION_CODES.get(op_code, f"Unknown_0x{op_code:02X}")

def filetime_to_datetime(filetime_bytes):
    """
    Convert FILETIME (8-byte little-endian) to datetime.
    
    Args:
        filetime_bytes: 8 bytes representing Windows FILETIME
        
    Returns:
        str: ISO format timestamp or None if invalid
    """
    if not filetime_bytes or len(filetime_bytes) != 8:
        return None
    
    try:
        # Unpack as 64-bit unsigned integer (little-endian)
        filetime = struct.unpack('<Q', filetime_bytes)[0]
        
        # FILETIME epoch: January 1, 1601
        # Convert to Unix epoch (January 1, 1970)
        EPOCH_DIFF = 116444736000000000  # 100-nanosecond intervals
        
        if filetime == 0:
            return None
        
        # Convert to microseconds for Python datetime
        microseconds = (filetime - EPOCH_DIFF) // 10
        
        # Create datetime object
        dt = datetime(1970, 1, 1) + timedelta(microseconds=microseconds)
        
        # Format with microsecond precision
        return dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    
    except (struct.error, ValueError, OSError, OverflowError):
        return None

def extract_timestamps_from_buffer(data_buffer, attribute_offset):
    """
    Extract timestamps from redo/undo data buffer based on attribute offset.
    
    Args:
        data_buffer: Raw bytes from redo_data or undo_data
        attribute_offset: Position within $SI attribute
        
    Returns:
        dict: Dictionary with C, M, E, A timestamps
    """
    timestamps = {
        "C": None,
        "M": None,
        "E": None,
        "A": None
    }
    
    if not data_buffer or len(data_buffer) < 8:
        return timestamps
    
    try:
        # Attribute offset determines which timestamps are present
        if attribute_offset == 0x18:
            # All 4 timestamps: C, M, E, A (32 bytes total)
            if len(data_buffer) >= 32:
                timestamps["C"] = filetime_to_datetime(data_buffer[0:8])
                timestamps["M"] = filetime_to_datetime(data_buffer[8:16])
                timestamps["E"] = filetime_to_datetime(data_buffer[16:24])
                timestamps["A"] = filetime_to_datetime(data_buffer[24:32])
        
        elif attribute_offset == 0x20:
            # M, E, A (24 bytes)
            if len(data_buffer) >= 24:
                timestamps["M"] = filetime_to_datetime(data_buffer[0:8])
                timestamps["E"] = filetime_to_datetime(data_buffer[8:16])
                timestamps["A"] = filetime_to_datetime(data_buffer[16:24])
        
        elif attribute_offset == 0x28:
            # E, A (16 bytes)
            if len(data_buffer) >= 16:
                timestamps["E"] = filetime_to_datetime(data_buffer[0:8])
                timestamps["A"] = filetime_to_datetime(data_buffer[8:16])
        
        elif attribute_offset == 0x30:
            # A only (8 bytes)
            if len(data_buffer) >= 8:
                timestamps["A"] = filetime_to_datetime(data_buffer[0:8])
    
    except Exception:
        pass
    
    return timestamps

def is_timestamp_change_record(record):
    """
    Check if a LogFile record represents a timestamp change event.
    Based on Algorithm 1 from Oh et al.
    
    Args:
        record: NTFSLogRecord object
        
    Returns:
        bool: True if this is a timestamp change record
    """
    try:
        # Check for UpdateResidentValue operation (0x7)
        if record.get_redo_operation() != 0x07:
            return False
        
        # Check for $STANDARD_INFORMATION attribute (offset 0x38)
        record_offset = record.get_record_offset()
        if record_offset != 0x38:
            return False
        
        # Check attribute offset is in timestamp range
        attr_offset = record.get_attribute_offset()
        if attr_offset < 0x18 or attr_offset > 0x30:
            return False
        
        return True
    
    except Exception:
        return False

print("Helper functions and constants defined successfully.")


Helper functions and constants defined successfully.


In [18]:
# [Cell 5] Parse LogFile and Extract Records

def parse_logfile(logfile_path, progress_interval=5000):
    """
    Parse the LogFile and extract all relevant transaction records.
    
    Args:
        logfile_path: Path to the $LogFile
        progress_interval: Print progress every N records
        
    Returns:
        list: List of dictionaries containing parsed records
    """
    records = []
    
    print(f"Opening LogFile: {logfile_path}")
    
    with open(logfile_path, "rb") as logfile:
        parser = LogFileParser(logfile)
        
        print("Collecting LSNs...")
        parser.collect_lsns()
        
        record_count = 0
        timestamp_change_count = 0
        error_count = 0
        
        print("Parsing LogFile records...")
        
        for record in parser.parse_ntfs_records():
            try:
                record_count += 1
                
                # Progress indicator
                if record_count % progress_interval == 0:
                    print(f"  Processed {record_count:,} records (timestamp changes: {timestamp_change_count:,})...")
                
                # Get basic record info
                lsn = record.lsn
                redo_op = record.get_redo_operation()
                undo_op = record.get_undo_operation()
                redo_op_name = get_operation_name(redo_op)
                undo_op_name = get_operation_name(undo_op)
                
                # Get target information
                try:
                    record_offset = record.get_record_offset()
                except Exception:
                    record_offset = None
                
                try:
                    attribute_offset = record.get_attribute_offset()
                except Exception:
                    attribute_offset = None
                
                try:
                    target_vcn = record.get_target_vcn()
                except Exception:
                    target_vcn = None
                
                try:
                    mft_target_number = record.calculate_mft_target_number()
                except Exception:
                    mft_target_number = None
                
                # Get redo and undo data
                try:
                    redo_data = record.get_redo_data()
                except Exception:
                    redo_data = None
                
                try:
                    undo_data = record.get_undo_data()
                except Exception:
                    undo_data = None
                
                # Check if this is a timestamp change record
                is_timestamp_change = is_timestamp_change_record(record)
                
                # Extract timestamps if this is a timestamp change
                undo_timestamps = {"C": None, "M": None, "E": None, "A": None}
                redo_timestamps = {"C": None, "M": None, "E": None, "A": None}
                
                if is_timestamp_change and attribute_offset is not None:
                    timestamp_change_count += 1
                    if undo_data:
                        undo_timestamps = extract_timestamps_from_buffer(undo_data, attribute_offset)
                    if redo_data:
                        redo_timestamps = extract_timestamps_from_buffer(redo_data, attribute_offset)
                
                # Create record
                parsed_record = {
                    "LSN": lsn,
                    "RedoOP": redo_op,
                    "UndoOP": undo_op,
                    "RedoOPName": redo_op_name,
                    "UndoOPName": undo_op_name,
                    "RecordOffset": record_offset,
                    "AttributeOffset": attribute_offset,
                    "TargetVCN": target_vcn,
                    "MFTTargetNumber": mft_target_number,
                    "IsTimestampChange": is_timestamp_change,
                    "Undo_SI_C": undo_timestamps["C"],
                    "Undo_SI_M": undo_timestamps["M"],
                    "Undo_SI_E": undo_timestamps["E"],
                    "Undo_SI_A": undo_timestamps["A"],
                    "Redo_SI_C": redo_timestamps["C"],
                    "Redo_SI_M": redo_timestamps["M"],
                    "Redo_SI_E": redo_timestamps["E"],
                    "Redo_SI_A": redo_timestamps["A"]
                }
                
                records.append(parsed_record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"  Warning: Error parsing record {record_count}: {str(e)[:100]}")
                elif error_count == 6:
                    print("  (Suppressing further error messages...)")
    
    print(f"\nParsing complete!")
    print(f"  Total records processed: {record_count:,}")
    print(f"  Timestamp change records: {timestamp_change_count:,}")
    print(f"  Records with errors: {error_count:,}")
    print(f"  Successfully parsed: {len(records):,}")
    
    return records

# Execute parsing
logfile_records = parse_logfile(INPUT_LOGFILE)


Opening LogFile: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$LogFile
Parsing LogFile records...
  Processed 5,000 records (timestamp changes: 126)...
  Processed 10,000 records (timestamp changes: 226)...
  Processed 15,000 records (timestamp changes: 356)...
  Processed 20,000 records (timestamp changes: 484)...
  Processed 25,000 records (timestamp changes: 604)...
  Processed 30,000 records (timestamp changes: 718)...
  Processed 35,000 records (timestamp changes: 772)...
  Processed 40,000 records (timestamp changes: 847)...
  Processed 45,000 records (timestamp changes: 1,046)...
  Processed 50,000 records (timestamp changes: 1,209)...
  Processed 55,000 records (timestamp changes: 1,313)...
  Processed 60,000 records (timestamp changes: 1,635)...
  Processed 65,000 records (timestamp changes: 1,829)...
  Processed 70,000 records (timestamp changes: 1,934)...
  Processed 75,000 records (timestamp changes: 1,982)...
  Processed 80,000 records (timestamp changes:

In [19]:
# [Cell 6] Create DataFrame and Organize Columns

# Create DataFrame
df_logfile = pd.DataFrame(logfile_records)

# Define column order
column_order = [
    "LSN",
    "RedoOP",
    "UndoOP",
    "RedoOPName",
    "UndoOPName",
    "RecordOffset",
    "AttributeOffset",
    "TargetVCN",
    "MFTTargetNumber",
    "IsTimestampChange",
    "Undo_SI_C",
    "Undo_SI_M",
    "Undo_SI_E",
    "Undo_SI_A",
    "Redo_SI_C",
    "Redo_SI_M",
    "Redo_SI_E",
    "Redo_SI_A"
]

# Reorder columns
df_logfile = df_logfile[column_order]

# Rename for clarity
df_logfile.columns = [
    "LSN",
    "RedoOP",
    "UndoOP",
    "RedoOPName",
    "UndoOPName",
    "RecordOffset",
    "AttributeOffset",
    "TargetVCN",
    "TargetFRN",
    "IsTimestampChange",
    "Undo_$SI-C",
    "Undo_$SI-M",
    "Undo_$SI-E",
    "Undo_$SI-A",
    "Redo_$SI-C",
    "Redo_$SI-M",
    "Redo_$SI-E",
    "Redo_$SI-A"
]

print(f"DataFrame created with {len(df_logfile):,} records and {len(df_logfile.columns)} columns")
print(f"\nColumn names: {list(df_logfile.columns)}")


DataFrame created with 417,267 records and 18 columns

Column names: ['LSN', 'RedoOP', 'UndoOP', 'RedoOPName', 'UndoOPName', 'RecordOffset', 'AttributeOffset', 'TargetVCN', 'TargetFRN', 'IsTimestampChange', 'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A', 'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A']


In [20]:
# [Cell 7] Data Quality Summary

print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

# Basic statistics
print(f"\nTotal Records: {len(df_logfile):,}")
print(f"Timestamp Change Records: {df_logfile['IsTimestampChange'].sum():,}")

# Missing values
print("\nMissing Values:")
for col in df_logfile.columns:
    null_count = df_logfile[col].isnull().sum()
    null_pct = (null_count / len(df_logfile)) * 100
    if null_pct > 0:
        print(f"  {col}: {null_count:,} ({null_pct:.2f}%)")

# Sample data
print("\n" + "=" * 60)
print("SAMPLE RECORDS (First 5)")
print("=" * 60)
df_logfile.head()


DATA QUALITY SUMMARY

Total Records: 417,267
Timestamp Change Records: 32,435

Missing Values:
  TargetFRN: 228,502 (54.76%)
  Undo_$SI-C: 417,154 (99.97%)
  Undo_$SI-M: 410,403 (98.36%)
  Undo_$SI-E: 405,691 (97.23%)
  Undo_$SI-A: 384,898 (92.24%)
  Redo_$SI-C: 417,154 (99.97%)
  Redo_$SI-M: 410,403 (98.36%)
  Redo_$SI-E: 405,691 (97.23%)
  Redo_$SI-A: 384,897 (92.24%)

SAMPLE RECORDS (First 5)


,LSN,RedoOP,UndoOP,RedoOPName,UndoOPName,RecordOffset,AttributeOffset,TargetVCN,TargetFRN,IsTimestampChange,Undo_$SI-C,Undo_$SI-M,Undo_$SI-E,Undo_$SI-A,Redo_$SI-C,Redo_$SI-M,Redo_$SI-E,Redo_$SI-A
0,8713796027,21,22,SetBitsInNonresidentBitMap,ClearBitsInNonresidentBitMap,0,0,11,NaN,False,None,None,None,None,None,None,None,None
1,8713796039,0,3,Noop,DeallocateFileRecordSegment,0,0,95098,380395.0,False,None,None,None,None,None,None,None,None
2,8713796051,14,15,AddIndexEntryAllocation,DeleteIndexEntryAllocation,0,3008,1,NaN,False,None,None,None,None,None,None,None,None
3,8713796079,14,15,AddIndexEntryAllocation,DeleteIndexEntryAllocation,0,1408,0,NaN,False,None,None,None,None,None,None,None,None
4,8713796112,2,0,InitializeFileRecordSegment,Noop,0,0,95098,380395.0,False,None,None,None,None,None,None,None,None


In [21]:
# [Cell 8] Analyze Operation Distribution

print("=" * 60)
print("OPERATION DISTRIBUTION")
print("=" * 60)

# Top 20 most common operations
op_counts = df_logfile["RedoOPName"].value_counts().head(20)

print("\nTop 20 Most Common Redo Operations:")
print("Rank | Count     | Operation Name")
print("-" * 60)
for idx, (op_name, count) in enumerate(op_counts.items(), 1):
    pct = (count / len(df_logfile)) * 100
    print(f"{idx:3d}  | {count:8,} ({pct:5.2f}%) | {op_name}")

# Focus on timestamp-related operations
print("\n" + "=" * 60)
print("TIMESTAMP-RELATED OPERATIONS")
print("=" * 60)

update_resident = df_logfile[df_logfile["RedoOP"] == 0x07]
print(f"\nUpdateResidentValue (0x07) records: {len(update_resident):,}")

timestamp_changes = df_logfile[df_logfile["IsTimestampChange"] == True]
print(f"Confirmed timestamp change records: {len(timestamp_changes):,}")

if len(timestamp_changes) > 0:
    # Attribute offset distribution
    print("\nAttribute Offset Distribution (Timestamp Changes):")
    attr_offset_counts = timestamp_changes["AttributeOffset"].value_counts().sort_index()
    for offset, count in attr_offset_counts.items():
        if offset == 0x18:
            desc = "All 4 timestamps (C, M, E, A)"
        elif offset == 0x20:
            desc = "M, E, A"
        elif offset == 0x28:
            desc = "E, A"
        elif offset == 0x30:
            desc = "A only"
        else:
            desc = "Unknown"
        print(f"  0x{offset:02X}: {count:,} records - {desc}")


OPERATION DISTRIBUTION

Top 20 Most Common Redo Operations:
Rank | Count     | Operation Name
------------------------------------------------------------
  1  |   93,951 (22.52%) | ForgetTransaction
  2  |   83,112 (19.92%) | SetNewAttributeSizes
  3  |   53,303 (12.77%) | UpdateResidentValue
  4  |   40,891 ( 9.80%) | UpdateFileNameAllocation
  5  |   29,529 ( 7.08%) | UpdateNonresidentValue
  6  |   20,028 ( 4.80%) | SetBitsInNonresidentBitMap
  7  |   16,758 ( 4.02%) | UpdateMappingPairs
  8  |   15,326 ( 3.67%) | ClearBitsInNonresidentBitMap
  9  |   13,018 ( 3.12%) | Noop
 10  |   10,713 ( 2.57%) | DeleteIndexEntryAllocation
 11  |    8,686 ( 2.08%) | UpdateFileNameRoot
 12  |    6,652 ( 1.59%) | AddIndexEntryAllocation
 13  |    5,618 ( 1.35%) | DeleteAttribute
 14  |    4,640 ( 1.11%) | CreateAttribute
 15  |    4,356 ( 1.04%) | DeallocateFileRecordSegment
 16  |    3,477 ( 0.83%) | InitializeFileRecordSegment
 17  |    2,450 ( 0.59%) | ZeroEndOfFileRecord
 18  |    1,683 ( 0.4

In [22]:
# [Cell 9] Analyze Timestamp Changes

print("=" * 60)
print("TIMESTAMP CHANGE ANALYSIS")
print("=" * 60)

if len(timestamp_changes) > 0:
    # Count changes by timestamp type
    print("\nTimestamp Change Statistics:")
    
    # Check which timestamps were modified
    si_c_changes = timestamp_changes[
        (timestamp_changes["Undo_$SI-C"].notna()) & 
        (timestamp_changes["Redo_$SI-C"].notna()) &
        (timestamp_changes["Undo_$SI-C"] != timestamp_changes["Redo_$SI-C"])
    ]
    print(f"  $SI-C changes: {len(si_c_changes):,}")
    
    si_m_changes = timestamp_changes[
        (timestamp_changes["Undo_$SI-M"].notna()) & 
        (timestamp_changes["Redo_$SI-M"].notna()) &
        (timestamp_changes["Undo_$SI-M"] != timestamp_changes["Redo_$SI-M"])
    ]
    print(f"  $SI-M changes: {len(si_m_changes):,}")
    
    si_e_changes = timestamp_changes[
        (timestamp_changes["Undo_$SI-E"].notna()) & 
        (timestamp_changes["Redo_$SI-E"].notna()) &
        (timestamp_changes["Undo_$SI-E"] != timestamp_changes["Redo_$SI-E"])
    ]
    print(f"  $SI-E changes: {len(si_e_changes):,}")
    
    si_a_changes = timestamp_changes[
        (timestamp_changes["Undo_$SI-A"].notna()) & 
        (timestamp_changes["Redo_$SI-A"].notna()) &
        (timestamp_changes["Undo_$SI-A"] != timestamp_changes["Redo_$SI-A"])
    ]
    print(f"  $SI-A changes: {len(si_a_changes):,}")
    
    # Sample timestamp change records
    print("\n" + "=" * 60)
    print("SAMPLE TIMESTAMP CHANGES (First 5)")
    print("=" * 60)
    
    sample_cols = ["LSN", "TargetFRN", "Undo_$SI-C", "Redo_$SI-C", "Undo_$SI-M", "Redo_$SI-M"]
    if len(si_c_changes) > 0:
        print("\n$SI-C Changes:")
        display(si_c_changes[sample_cols].head())
    elif len(si_m_changes) > 0:
        print("\n$SI-M Changes:")
        display(si_m_changes[sample_cols].head())
else:
    print("\nNo timestamp change records found.")


TIMESTAMP CHANGE ANALYSIS

Timestamp Change Statistics:
  $SI-C changes: 113
  $SI-M changes: 6,766
  $SI-E changes: 10,597
  $SI-A changes: 26,913

SAMPLE TIMESTAMP CHANGES (First 5)

$SI-C Changes:


,LSN,TargetFRN,Undo_$SI-C,Redo_$SI-C,Undo_$SI-M,Redo_$SI-M
28640,8714363223,382342.0,2023-12-22 16:14:25.663566,2022-12-20 13:29:04.007776,2000-01-01 00:00:00.000000,2000-01-01 00:00:00.000000
28670,8714363824,382341.0,2023-12-22 16:14:25.663566,2022-12-20 13:29:03.992149,2023-12-18 20:21:20.000000,2023-12-18 20:21:20.000000
43685,8714632291,1504.0,2023-12-22 16:14:27.130275,2022-12-16 09:32:24.665083,2023-12-22 16:14:27.130275,2023-12-22 16:14:27.130275
48869,8714726227,84021.0,2023-12-22 16:14:29.599567,2023-12-18 20:19:22.000000,2023-12-18 20:19:22.000000,2023-12-18 20:19:22.000000
49087,8714730302,84026.0,2023-12-22 16:14:29.614814,2023-12-18 20:19:22.000000,2023-12-18 20:19:22.000000,2023-12-18 20:19:22.000000


In [23]:
# [Cell 10] Target File Analysis

print("=" * 60)
print("TARGET FILE ANALYSIS")
print("=" * 60)

# Unique target files
valid_targets = df_logfile[df_logfile["TargetFRN"].notna()]
unique_frns = valid_targets["TargetFRN"].nunique()

print(f"\nRecords with target FRN: {len(valid_targets):,}")
print(f"Unique target files: {unique_frns:,}")

if unique_frns > 0:
    print(f"Average records per file: {len(valid_targets) / unique_frns:.1f}")
    
    # Most active files
    file_activity = valid_targets["TargetFRN"].value_counts().head(10)
    
    print("\nTop 10 Files by Record Count:")
    print("Rank | Records   | File Reference Number")
    print("-" * 60)
    for idx, (frn, count) in enumerate(file_activity.items(), 1):
        print(f"{idx:3d}  | {count:8,} | {int(frn)}")

# Timestamp changes by file
if len(timestamp_changes) > 0:
    ts_change_files = timestamp_changes["TargetFRN"].value_counts()
    print(f"\nFiles with timestamp changes: {len(ts_change_files):,}")
    
    if len(ts_change_files) > 0:
        print(f"Max timestamp changes for a single file: {ts_change_files.max()}")


TARGET FILE ANALYSIS

Records with target FRN: 188,765
Unique target files: 22,717
Average records per file: 8.3

Top 10 Files by Record Count:
Rank | Records   | File Reference Number
------------------------------------------------------------
  1  |   58,540 | 103255
  2  |    3,695 | 6
  3  |    3,061 | 191670
  4  |    2,751 | 131284
  5  |    2,403 | 73028
  6  |    1,785 | 30516
  7  |    1,538 | 33693
  8  |    1,353 | 44293
  9  |    1,285 | 31950
 10  |    1,000 | 30492

Files with timestamp changes: 20,439
Max timestamp changes for a single file: 420


In [24]:
# [Cell 11] Save to CSV

# Save DataFrame to CSV
df_logfile.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# Verify the saved file
saved_size = OUTPUT_CSV.stat().st_size / (1024 * 1024)
print(f"LogFile data saved to: {OUTPUT_CSV}")
print(f"Output file size: {saved_size:.2f} MB")

# Verify by reading back
df_verify = pd.read_csv(OUTPUT_CSV)
print(f"Verification: {len(df_verify):,} records saved successfully")


LogFile data saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/LogFile.csv
Output file size: 36.55 MB
Verification: 417,267 records saved successfully


In [25]:
# [Cell 12] Final Summary and Statistics

print("=" * 60)
print("PHASE 1 - LOGFILE PARSING COMPLETE")
print("=" * 60)

timestamp_change_count = df_logfile["IsTimestampChange"].sum()

print(f"""
Input:  {INPUT_LOGFILE}
Output: {OUTPUT_CSV}

Statistics:
-----------
Total LogFile Records:          {len(df_logfile):,}
Timestamp Change Records:       {timestamp_change_count:,}
Unique Target Files:            {df_logfile['TargetFRN'].nunique():,}
UpdateResidentValue (0x07):     {len(df_logfile[df_logfile['RedoOP'] == 0x07]):,}

Columns Extracted:
------------------
{chr(10).join(f'  - {col}' for col in df_logfile.columns)}

Detection Patterns Ready:
-------------------------
Algorithm 1: Timestamp change events identified
Algorithm 2: Target FRN available for MFT cross-reference
Algorithm 3: Before/After timestamps for comparison
Algorithm 4: Ready for File System Tunneling analysis in Phase 2

Ready for Phase 2: Data Preprocessing and Event Grouping
""")


PHASE 1 - LOGFILE PARSING COMPLETE

Input:  /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$LogFile
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/LogFile.csv

Statistics:
-----------
Total LogFile Records:          417,267
Timestamp Change Records:       32,435
Unique Target Files:            22,717
UpdateResidentValue (0x07):     53,303

Columns Extracted:
------------------
  - LSN
  - RedoOP
  - UndoOP
  - RedoOPName
  - UndoOPName
  - RecordOffset
  - AttributeOffset
  - TargetVCN
  - TargetFRN
  - IsTimestampChange
  - Undo_$SI-C
  - Undo_$SI-M
  - Undo_$SI-E
  - Undo_$SI-A
  - Redo_$SI-C
  - Redo_$SI-M
  - Redo_$SI-E
  - Redo_$SI-A

Detection Patterns Ready:
-------------------------
Algorithm 1: Timestamp change events identified
Algorithm 2: Target FRN available for MFT cross-reference
Algorithm 3: Before/After timestamps for comparison
Algorithm 4: Ready for File System Tunneling analysis in Phase 2

Ready for Phase 2